# 03 — Speaker Segmentation and Role Assignment

**Goal:** turn the recognized words of notebook 02 into a readable conversation: who spoke, in what order, labelled **Speaker 1 / Speaker 2** and **Customer / Agent**.

**Runs on:** a laptop (no GPU needed).

**Input:** `results/asr_words.csv` (from notebook 02), `results/subset_ids.csv`, and the dataset metadata.

**Output**
- `results/segmented_turns.csv` — one row per speaker turn
- `results/transcripts/<call_id>.txt` — one readable conversation per call

**Sections**
1. Four related tasks, and which one we need
2. Setup
3. Build speaker turns
4. Speaker labels and roles
5. Unclear audio and overlapping speech
6. Checks against the reference
7. Save the results
8. Example conversations
9. Findings

## 1. Four related tasks, and which one we need

| Task | Question it answers | Do we need a model? |
|---|---|---|
| **Speaker segmentation** | *Where* does one speaker's speech start and stop? | No — each speaker has their own audio channel, and Whisper gives word timestamps |
| **Speaker diarization** | *Who spoke when*, when all voices are mixed in one recording? | No — the voices are already separated by the recording setup (two channels) |
| **Speaker identification** | *Which known person* is this voice? | Not needed for this task |
| **Role assignment** | Is this speaker the **Customer** or the **Agent**? | No — taken from the dataset metadata, then **checked** against what each speaker says |

**Why no diarization model?** A diarization model (for example pyannote) would have to *guess* who is speaking from the sound of the voice, and it makes mistakes when people talk at once or sound similar. Our recordings already keep the two speakers on separate channels, so speaker separation is given by the data. We use that, and spend our effort on **checking** it.

**Labels we use**
- **Speaker 1 / Speaker 2** = audio channel 1 / channel 2.
- **Customer / Agent** = the role of that channel in the transcript metadata (`caller` → Customer). Notebook 01 showed we must **not** assume Speaker 1 is always the customer: in 37 calls the agent is on channel 1.

## 2. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import RESULTS_DIR, build_utterances_table
from src.segmentation import (format_conversation, role_from_content,
                              utterances_to_turns, words_to_utterances)

MAX_PAUSE_S = 1.0          # a pause longer than this starts a new utterance
LOW_CONFIDENCE = 0.5       # turns whose mean word probability is below this are flagged

pd.set_option("display.max_colwidth", 100)

subset = pd.read_csv(RESULTS_DIR / "subset_ids.csv")
asr_words = pd.read_csv(RESULTS_DIR / "asr_words.csv")
reference = build_utterances_table(subset["call_id"].tolist())   # used only for checks in section 6

print(f"Calls: {asr_words['call_id'].nunique()}  |  recognized words: {len(asr_words)}"
      f"  |  ASR model: {asr_words['asr_model'].iloc[0]}")

Calls: 25  |  recognized words: 2255  |  ASR model: medium.en


## 3. Build speaker turns

Two steps, both in `src/segmentation.py`:

1. **Words → utterances (per speaker):** consecutive words of the same speaker stay together until a pause longer than `MAX_PAUSE_S` (1 second). We use word timestamps, because one Whisper *segment* can cover several sentences and the pauses between them.
2. **Utterances → turns (whole call):** all utterances of both speakers are sorted by start time. Neighbouring utterances from the same speaker are merged into one **turn**, so the conversation reads naturally: Agent, Customer, Agent, ...

In [2]:
utterances = words_to_utterances(asr_words, max_pause_s=MAX_PAUSE_S)
turns = utterances_to_turns(utterances)

print(f"Utterances: {len(utterances)}  |  turns: {len(turns)}  |  "
      f"median turns per call: {turns.groupby('call_id').size().median():.0f}")
turns[["call_id", "turn_index", "speaker", "role", "start_s", "end_s", "text"]].head(8)

Utterances: 283  |  turns: 237  |  median turns per call: 9


,call_id,turn_index,speaker,role,start_s,end_s,text
0,08f360b3f3b844ef,1,Speaker 2,Agent,1.84,7.26,"Hello, this is Harbor Valley National Bank. My name is Robert. How can I help you today?"
1,08f360b3f3b844ef,2,Speaker 1,Customer,10.89,14.59,Hi. My name is Linda Smith. I need to check my account balance.
2,08f360b3f3b844ef,3,Speaker 2,Agent,14.50,24.04,"Sure, I'd be happy to do that for you. And what's your name, please? Oh, right, it's Linda Smith..."
3,08f360b3f3b844ef,4,Speaker 1,Customer,23.10,24.14,Linda Smith.
4,08f360b3f3b844ef,5,Speaker 2,Agent,25.32,30.02,"Alright, I'm in. And which account would you like to check? Your savings account or your checkin..."
5,08f360b3f3b844ef,6,Speaker 1,Customer,25.84,35.58,"All is well. I'd like to check my checking account, please."
6,08f360b3f3b844ef,7,Speaker 2,Agent,42.20,45.32,Your checking account balance is $90 exactly.
7,08f360b3f3b844ef,8,Speaker 1,Customer,49.76,51.24,Fantastic. That's all I needed.


## 4. Speaker labels and roles

The role comes from the metadata. To **check** it independently, `role_from_content` guesses the agent from what is said: in this dataset only the bank agent opens with phrases such as *"harper valley"* or *"how can I help"*. If both methods agree, we can trust the labels.

In [3]:
content_guess = role_from_content(turns)
metadata_agent = turns[turns["role"] == "Agent"].groupby("call_id")["channel_index"].first()

role_check = pd.DataFrame({
    "agent_channel_metadata": metadata_agent,
    "agent_channel_from_content": pd.Series(content_guess),
}).reset_index(names="call_id")
role_check["agree"] = role_check["agent_channel_metadata"] == role_check["agent_channel_from_content"]
role_check["content_guess_found"] = role_check["agent_channel_from_content"].notna()

found = role_check[role_check["content_guess_found"]]
print(f"Calls where the content check found the agent: {len(found)} of {len(role_check)}")
print(f"... and agrees with the metadata: {found['agree'].sum()} of {len(found)}")
role_check.merge(subset[["call_id", "selection_reason"]], on="call_id").sort_values("agree")

Calls where the content check found the agent: 22 of 25
... and agrees with the metadata: 22 of 22


,call_id,agent_channel_metadata,agent_channel_from_content,agree,content_guess_found,selection_reason
1,09fc75fc02ea4b46,2,NaN,False,False,stratified random
18,a35cc473b91f4de0,2,NaN,False,False,stratified random
14,8cee43b11a144b7d,2,NaN,False,False,stratified random
0,08f360b3f3b844ef,2,2.0,True,True,stratified random
22,cb7bc09bb8d440dc,2,2.0,True,True,stratified random
21,c0c8d261058841ac,2,2.0,True,True,stratified random
20,c09408163cb14e39,2,2.0,True,True,stratified random
19,b711d5c6db68429b,2,2.0,True,True,stratified random
17,9e05bd9ca9564089,2,2.0,True,True,stratified random
16,96c25385230d4172,2,2.0,True,True,stratified random


## 5. Unclear audio and overlapping speech

We do not hide uncertain output. Each turn gets two flags:
- **`low_confidence`** — the mean Whisper word probability is below 0.5: the audio was hard to understand, so the text may be wrong.
- **`overlaps_other_speaker`** — both people talked at the same time for more than half a second. Common in real calls (for example "mm-hmm" while the other person talks). With separate channels both voices are still transcribed; the flag just marks that the order of those two turns is approximate.

In [4]:
turns["low_confidence"] = turns["mean_word_probability"] < LOW_CONFIDENCE

flags = pd.DataFrame({
    "turns": [len(turns)],
    "low_confidence": [int(turns["low_confidence"].sum())],
    "overlaps_other_speaker": [int(turns["overlaps_other_speaker"].sum())],
})
print(flags.to_string(index=False))
turns.loc[turns["low_confidence"], ["call_id", "role", "start_s", "text", "mean_word_probability"]].round(2).head(10)

 turns  low_confidence  overlaps_other_speaker
   237               3                      55


,call_id,role,start_s,text,mean_word_probability
11,08f360b3f3b844ef,Customer,60.52,useful,0.38
202,cb7bc09bb8d440dc,Customer,20.99,Saturday,0.29
210,d3baa07088cf4c76,Customer,16.75,"Hello,",0.48


## 6. Checks against the reference

The human reference is used **only to evaluate**, never to build our output.

**Check 1 — speaker attribution:** notebook 02 marked every recognized word that falls inside a reference utterance of the **same** speaker (`in_own_speech`). The share of such words tells us how often a word is attributed to the right speaker.

**Check 2 — turn structure:** we build turns from the reference utterances with the same merge rule and compare the number of turns per call.

In [5]:
words_with_text = asr_words[asr_words["has_text"]]
attribution = words_with_text["in_own_speech"].mean()
print(f"Check 1 — words inside the same speaker's reference speech: {attribution:.1%}")

ref_speech = reference[~reference["is_empty"] & ~reference["is_noise_only"]].rename(
    columns={"human_transcript": "text"})
ref_speech = ref_speech.assign(n_words=ref_speech["text"].str.split().str.len(), mean_word_probability=1.0)
ref_turns = utterances_to_turns(ref_speech[["call_id", "channel_index", "speaker_role", "start_s", "end_s",
                                            "text", "n_words", "mean_word_probability"]])

turn_counts = pd.DataFrame({"asr_turns": turns.groupby("call_id").size(),
                            "reference_turns": ref_turns.groupby("call_id").size()})
turn_counts["difference"] = turn_counts["asr_turns"] - turn_counts["reference_turns"]
print(f"Check 2 — calls with the same number of turns: {(turn_counts['difference'] == 0).sum()} of {len(turn_counts)}"
      f"  |  mean absolute difference: {turn_counts['difference'].abs().mean():.1f} turns")
turn_counts.sort_values("difference", key=abs, ascending=False).head(8)

Check 1 — words inside the same speaker's reference speech: 99.4%
Check 2 — calls with the same number of turns: 15 of 25  |  mean absolute difference: 0.6 turns


,asr_turns,reference_turns,difference
call_id,,,
0f87dc392db84250,9,11,-2
663201e33f1a4e38,9,11,-2
d3baa07088cf4c76,20,22,-2
8cee43b11a144b7d,14,16,-2
96c25385230d4172,8,10,-2
c09408163cb14e39,9,11,-2
8b04254fc6b447b1,12,11,1
cb7bc09bb8d440dc,10,9,1


## 7. Save the results

In [6]:
output_cols = ["call_id", "turn_index", "speaker", "role", "start_s", "end_s", "text", "n_words",
               "mean_word_probability", "low_confidence", "overlaps_other_speaker"]
turns[output_cols].round({"start_s": 2, "end_s": 2, "mean_word_probability": 3}).to_csv(
    RESULTS_DIR / "segmented_turns.csv", index=False)

transcript_dir = RESULTS_DIR / "transcripts"
transcript_dir.mkdir(exist_ok=True)
for call_id, call_turns in turns.groupby("call_id"):
    (transcript_dir / f"{call_id}.txt").write_text(format_conversation(call_turns) + "\n")

print(f"Saved {len(turns)} turns and {turns['call_id'].nunique()} transcript files.")

Saved 237 turns and 25 transcript files.


## 8. Example conversations

A normal call, and the **edge-case** call chosen in notebook 01 (agent on channel 1). In the edge case, Speaker 1 must be labelled **Agent**.

In [7]:
normal_call = subset.loc[subset["selection_reason"] == "stratified random", "call_id"].iloc[0]
edge_calls = subset.loc[subset["agent_on_channel_1"], "call_id"].tolist()

for call_id in [normal_call] + edge_calls[:1]:
    print(f"===== {call_id} =====")
    print(format_conversation(turns[turns["call_id"] == call_id]))
    print()

===== 08f360b3f3b844ef =====
[   1.8s] Speaker 2 (Agent): Hello, this is Harbor Valley National Bank. My name is Robert. How can I help you today?
[  10.9s] Speaker 1 (Customer): Hi. My name is Linda Smith. I need to check my account balance.
[  14.5s] Speaker 2 (Agent): Sure, I'd be happy to do that for you. And what's your name, please? Oh, right, it's Linda Smith. Sorry about that. Linda Smith, right.
[  23.1s] Speaker 1 (Customer): Linda Smith.
[  25.3s] Speaker 2 (Agent): Alright, I'm in. And which account would you like to check? Your savings account or your checking account?
[  25.8s] Speaker 1 (Customer): All is well. I'd like to check my checking account, please.
[  42.2s] Speaker 2 (Agent): Your checking account balance is $90 exactly.
[  49.8s] Speaker 1 (Customer): Fantastic. That's all I needed.
[  50.4s] Speaker 2 (Agent): Great. Is there anything else I could help you with today?
[  55.8s] Speaker 1 (Customer): Nope. That'll be everything. Thank you very much.
[  56.6s] 

## 9. Findings

**Speaker separation and roles**
- Speakers are separated by the recording itself (one audio channel per speaker), so no diarization model was needed.
- Roles come from the dataset metadata. An independent content check (who says the bank greeting) found the agent in 22 of 25 calls and agreed with the metadata in all 22.
- The check failed in 3 calls mainly because Whisper wrote "Harbor Valley" instead of "Harper Valley": a domain-name recognition error. Giving Whisper the expected bank name as a hint is a possible improvement.
- The edge-case call (agent on channel 1) is labelled correctly: Speaker 1 = Agent.

**Checks against the reference**
- 99.4% of recognized words fall inside the same speaker's reference speech (correct attribution).
- 15 of 25 calls have exactly the same number of turns as the reference; the mean difference is 0.6 turns.

**Unclear audio and overlap**
- 3 of 237 turns have low confidence (mean word probability < 0.5). All are single words, which are harder to recognize without context.
- 55 turns overlap the other speaker by more than 0.5 s. Their order in the transcript is approximate.

**Limitations**
- The order of overlapping turns is based on start time only.
- The 1-second pause rule for splitting turns was set by hand, not tuned.